# 03c — The Self-Adjoint Hamiltonian (C)

**Requires the C kernel.** Install with `./install_c_kernel.sh`.

This notebook covers:
- RedBlue Geometries Engine: the RedBlue Hamiltonian and its two components
- Self-adjoint condition: R̂†_p = B̂_p
- The four values of d* — all necessary to perform ln(10) in radial complex spherical polar space
- GAP = 0.000707: derived from the d* gap, not empirical
- σ projections at different strata

Parallel Python notebook: `../03_self_adjoint_hamiltonian.ipynb`

## 1. RedBlue Geometries Engine — The Hamiltonian

```
RedBlue Geometries Engine = Σ_p  p^{-σ}  [R̂_p ⊗ ∂̂_∂M  +  ∂̂†_∂M ⊗ B̂_p]

R̂_p = xp                           Berry-Keating (what it IS)
B̂_p = ½p² + ℘(x; g₂, g₃)          Fermat-Weierstrass (what it CANNOT BE)
R̂†_p = B̂_p                         functional equation as operator identity
```

Self-adjoint: `H† = H`. The adjoint of the forward channel is the backward channel.  
The operator contains its own inverse.

In [ ]:
#include <stdio.h>
#include <math.h>

/* R_hat_p: Berry-Keating component (what it IS) */
static double R_hat(double x, double p) {
    return x * p;   /* xp in operator algebra */
}

/* B_hat_p: Fermat-Weierstrass component (what it CANNOT BE) */
/* Simplified: half-p-squared term only (Weierstrass P omitted for demo) */
static double B_hat(double p) {
    return 0.5 * p * p;
}

/* Geometric coupling */
static double G_p(int p, double sigma) {
    return pow((double)p, -sigma);
}

int main(void) {
    static const int primes[] = {2, 3, 5, 7, 11, 13};
    double x = 1.5, p_val = 0.3;
    double sigma = 0.5;   /* critical line */

    printf("RedBlue Geometries Engine components at sigma=1/2, x=%.1f, p=%.1f:\n\n", x, p_val);
    printf("  %-4s  %10s  %10s  %10s  %10s\n",
           "p", "G_p(1/2)", "R_hat", "B_hat", "R†=B?");
    printf("  %-4s  %10s  %10s  %10s  %10s\n",
           "----", "----------", "----------", "----------", "----------");
    for (int i = 0; i < 6; i++) {
        int p  = primes[i];
        double g  = G_p(p, sigma);
        double rh = R_hat(x, (double)p);
        double bh = B_hat((double)p);
        /* Self-adjoint check: R† = B means the adjoint of xp is 1/2 p^2 */
        /* In the full operator: this is the functional equation ξ(s)=ξ(1-s) */
        printf("  p=%-2d  %10.6f  %10.4f  %10.4f  %s\n",
               p, g, rh, bh,
               (rh > 0 && bh > 0) ? "consistent" : "?");
    }
    printf("\nR_hat = xp (Berry-Keating): what IS at each prime address.\n");
    printf("B_hat = 1/2 p^2 + P(x): what CANNOT BE (Fermat-Weierstrass).\n");
    printf("Self-adjoint: the operator returns the same truth from both directions.\n");
    return 0;
}

## 2. Self-Adjoint: Truth-Preserving, Not Form-Preserving

Self-adjoint does **not** mean form-preserving. It means truth-preserving across representations.

```
"A! = A"    (constraint — selects A=1)
```

This is self-adjoint: the operator returns the same truth from any representation.  
Not `"1 = 1"` — that describes a result. The self-adjoint operator is the **constraint** that gets there.

In [ ]:
#include <stdio.h>
#include <math.h>

/* Functional equation of zeta: xi(s) = xi(1-s)
 * Fixed point: s = 1-s  ->  s = 1/2.
 * This IS the self-adjoint condition for RedBlue Geometries Engine. */

static double xi_approx(double sigma, double E) {
    /* Simplified: E_red and E_blue from Noether current */
    double E_red  = exp(-sigma * E);
    double E_blue = exp(-(1.0 - sigma) * E);
    return E_red - E_blue;   /* = 0 at sigma=1/2 */
}

int main(void) {
    double E = 0.40;
    printf("Functional equation balance  xi(sigma) - xi(1-sigma):\n");
    printf("(= Noether current J(sigma, E=%.2f))\n\n", E);
    printf("  %-8s  %14s  %s\n", "sigma", "J = xi - xi*", "");
    printf("  %-8s  %14s\n", "--------", "-------------");
    for (int i = 0; i <= 10; i++) {
        double s = i / 10.0;
        double j = xi_approx(s, E);
        printf("  sigma=%.1f  %14.6e  %s\n", s, j,
               fabs(j) < 1e-10 ? "<-- self-adjoint fixed point" : "");
    }
    printf("\nThe fixed point of xi(s)=xi(1-s) is s=1/2.\n");
    printf("This is sigma=1/2 — the critical line — forced by the Hamiltonian.\n");
    return 0;
}

## 3. The Four Values of d*

d* is a 4-component object in radial complex spherical polar algebra space — one component per Cayley-Dickson stratum (ℝ, ℂ, ℍ, 𝕆).

**All four are necessary to perform ln(10) in that space.**

| Symbol | Name | Value | Role |
|---|---|---|---|
| d* | The Boundary | 0.24600 | BK spectral coordinate — σ=½ fixed point. ℂ-projection. |
| d*_taut | The Flow | Ω/ln(10) = 0.24631… | Tautological ceiling — d*_taut × ln(10) = Ω exactly. |
| d*_ln(10) | The Translator | d* × ln(10) = 0.56644… | Bridge to decimal scale. |
| d*_RG | The Stability | 0.24682 | Renormalization group flow fixed point. |

```
gap = |Ω − d* × ln(10)| = |0.56714 − 0.56644| = 0.000707 = GAP
```

GAP is not empirical — it is the d* gap. The ℝ, ℍ, and 𝕆 strata contribute the gap; 0.24600 (the ℂ-projection alone) cannot reach Ω.

In [ ]:
#include <stdio.h>
#include <math.h>

#define D_STAR     0.24600          /* The Boundary — C-projection of d* */
#define D_STAR_RG  0.24682          /* The Stability — RG fixed point */
#define OMEGA_ZS   0.56714329       /* Lambert W fixed point W(1) */

int main(void) {
    double ln10     = log(10.0);
    double d_taut   = OMEGA_ZS / ln10;          /* The Flow */
    double d_ln10   = D_STAR * ln10;            /* The Translator */
    double gap      = fabs(OMEGA_ZS - d_ln10);  /* = GAP */

    printf("The four values of d* (all necessary for ln(10) in\n");
    printf("radial complex spherical polar coordinate space):\n\n");
    printf("  d*          (The Boundary)   = %.5f   [C-projection, used in word_coords()]\n", D_STAR);
    printf("  d*_taut     (The Flow)       = %.5f   [Omega/ln(10) = %.5f/%.5f]\n",
           d_taut, OMEGA_ZS, ln10);
    printf("  d*_ln(10)   (The Translator) = %.5f   [d* x ln(10)]\n", d_ln10);
    printf("  d*_RG       (The Stability)  = %.5f   [RG flow fixed point]\n", D_STAR_RG);
    printf("\n");
    printf("  gap = |Omega - d* x ln(10)|\n");
    printf("      = |%.5f - %.5f|\n", OMEGA_ZS, d_ln10);
    printf("      = %.6f\n", gap);
    printf("\n  gap = 0.000707 = GAP  (Yang-Mills mass gap / J^mu clamp)\n");
    printf("\n  The gap is the contribution of the R, H, O strata of d*.\n");
    printf("  0.24600 (C-projection alone) cannot reach Omega without the gap.\n");
    printf("  The full octonionic measure over all four strata generates ln(10).\n");
    return 0;
}

## 4. σ Projections at Different Strata

RedBlue Geometries Engine has different projections at different σ values. Same operator, different representations.

In [ ]:
#include <stdio.h>
#include <math.h>

static const int PRIMES[] = {2, 3, 5, 7, 11};

static double H_contribution(int p, double sigma, double x, double p_val) {
    double G   = pow((double)p, -sigma);
    double R   = x * p_val;            /* R_hat = xp */
    double B   = 0.5 * p_val * p_val;  /* B_hat (partial) */
    return G * (R + B);
}

int main(void) {
    double x = 1.5, p_val = 0.3;
    double sigmas[] = {0.0, 0.5, 1.0, 2.0};
    const char *labels[] = {
        "sigma=0   (ground state, R, all G_p=1)",
        "sigma=1/2 (critical line, C, Monad)",
        "sigma=1   (Standard Model, H, Yang-Mills)",
        "sigma=2   (beyond SM, O, extreme suppression)"
    };

    printf("RedBlue Geometries Engine projection at each sigma (x=%.1f, p=%.1f):\n\n", x, p_val);
    for (int si = 0; si < 4; si++) {
        double s = sigmas[si];
        double H_sum = 0.0;
        for (int i = 0; i < 5; i++)
            H_sum += H_contribution(PRIMES[i], s, x, p_val);
        printf("  %-46s  H_sum = %.6f\n", labels[si], H_sum);
    }
    printf("\nSame operator. Different projections. Different physics.\n");
    printf("sigma=1/2 is the self-adjoint fixed point — Noether balance.\n");
    return 0;
}

## 5. Version Output — Constants in the Binary

The `-V` flag prints all engine constants. Run against your ptolemy binary:

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    system(PTOLEMY " -V 2>/dev/null");
    return 0;
}

## Summary

- RedBlue Geometries Engine = Σ_p p^{-σ} [R̂_p ⊗ ∂̂ + ∂̂† ⊗ B̂_p]. Self-adjoint: R̂†_p = B̂_p.
- Self-adjoint means truth-preserving, not form-preserving. The constraint `A! = A` selects A=1 — it does not state `1=1`.
- σ=½ is the functional equation fixed point — the unique self-adjoint point.
- d* is a 4-component object. All 4 values are required to perform ln(10) in radial complex spherical polar space.
- GAP = 0.000707 = gap = |Ω − d*×ln(10)|. Derived, not empirical.

**Next:** `04c_lagrangian_learning.ipynb` — β deepening and the A-coupling field.